## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到所在的代码树根 (`solutions/` 或 `tutorials/`)，
   这样 `from attention.mha import ...` 这种导入能直接生效。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd to the tree root (whichever of `solutions/` or `tutorials/` this notebook lives in), turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys

# Walk up from the notebook's CWD until we find a directory named
# `solutions` or `tutorials`. Works no matter which tree the student
# opened. 不论 notebook 位于 solutions/ 还是 tutorials/ 都能正确定位。
ROOTS = {'solutions', 'tutorials'}
if os.path.basename(os.getcwd()) not in ROOTS:
    while os.path.basename(os.getcwd()) not in ROOTS and os.getcwd() != '/':
        os.chdir('..')
    if os.path.basename(os.getcwd()) not in ROOTS:
        # Fallback: maybe we were started at the repo root.
        if os.path.isdir('tutorials'):
            os.chdir('tutorials')
        elif os.path.isdir('solutions'):
            os.chdir('solutions')

assert os.path.basename(os.getcwd()) in ROOTS, (
    f'could not locate solutions/ or tutorials/ from {os.getcwd()}')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the feature_embedding chapter's reference .pt files live
control_folder = 'feature_embedding/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 3 章 · Feature embedding

## 这一章在干啥

前面两章你写的全是"如何把张量变成另一张量"的几何 / 注意力运算。但 AF3 真实输入根本不是张量 —— 是**异质数据**:

- 蛋白序列 (一串字符)
- 多重序列对齐 (MSA, 不定行不定列)
- 同源模板结构 (3D 坐标 + atom mask)
- 化学组分 (CCD codes、原子电荷、原子名)
- 实验约束 (用户给的 contact / pocket / bond)
- 扩散里的噪声水平 (一个标量 σ)

**Feature embedding 这一章负责把异质 → 张量**。但工程量太大 (有 InputFeatureEmbedder /AtomAttentionEncoder / ConstraintEmbedder 等好几个 wrapper)，单元测试也难写(它们的输入是一整个 feature dict 而非简单张量)。

我们在这一章只单测**两块通用基础设施**：每对 token 的相对位置编码、和标量到向量的Fourier 嵌入。它们小而独立，是其它 embedder 内部反复调用的零件。其余更重的embedder 在端到端 notebook (`model/overview.ipynb`) 里整体验证。

## 本章模块

| 文件 | 类 / 函数 | 算法 |
|---|---|---|
| `relative_position_encoding.py` | `RelativePositionEncoding.generate_relp` + `.forward` | Algorithm 3 |
| `relative_position_encoding.py` | `FourierEmbedding` | Algorithm 22 |

## 3.1 RelativePositionEncoding (Algorithm 3)

Pairformer 和 DiffusionConditioning 都需要在初始 pair 张量上**叠加一个相对位置先验**:

「两个 token 在同一条链且只差 1 个残基」与「两个 token 来自完全不同的实体」这两种 pair 应该在初始时就有不同表示。绝对位置 (positional encoding) 在多链系统里没意义 (链是无序的) —— **相对位置才有意义**。

### Algorithm 3 的拼图

对每对 token (i, j) 算 3 类整数偏移，每类 clip 到一个固定范围、用一个特殊编码(`2*r_max+1` 或 `2*s_max+1`) 表示「超出范围或不同 chain」:

1. **residue 偏移** (gated by 同链): `clip(residue_index[i] - residue_index[j] + r_max, 0, 2r_max)`
2. **token 偏移** (gated by 同链 ∧ 同残基): 同样 clip，但 gating 更严格 ——   只在「同一个残基里的不同 token」(比如修饰残基/糖基/多原子 token) 才有非平凡值
3. **chain 偏移** (gated by 同 entity，`s_max`-clip): 给出蛋白寡聚体里第几条   对称拷贝

三类各自 one-hot，再拼上 1 维 `same_entity` 布尔，**总宽度 4·r_max + 2·s_max + 7**。

### forward 部分

用一个 `LinearNoBias((4*r_max + 2*s_max + 7), c_z)` 把上面的 one-hot 投到 pair 通道。这一步是可学的，作为整个 trunk 的「相对位置感知」起点。

### 关于 `generate_relp`

为什么是"generate"而不是"compute on the fly"? AF3 推理在每个 N_cycle 里都需要 relp，但 relp 是和 t 无关的常量。所以我们一次性算好塞回 `input_feature_dict`，后续 cycle 直接读。注意整段写在 `torch.no_grad()` 里 —— relp 无梯度。

**任务**: 在 `feature_embedding/relative_position_encoding.py` 填 `generate_relp` 的TODO 块。`forward` 仅是 `self.linear_no_bias(relp_feature)` 一行，不需要你新写。

In [ ]:
from feature_embedding.relative_position_encoding import RelativePositionEncoding
from feature_embedding.control_values.feature_embedding_checks import (
    r_max, s_max, c_z, test_inputs, test_module_shape, test_module_forward,
)

relpe = RelativePositionEncoding(r_max=r_max, s_max=s_max, c_z=c_z)
test_module_shape(relpe, 'relative_position_encoding', control_folder)
test_module_forward(
    relpe, 'relative_position_encoding',
    inputs=(test_inputs['relp_feature'],),
    output_names='out',
    control_folder=control_folder,
)
print('RelativePositionEncoding ✓')

## 3.2 FourierEmbedding (Algorithm 22)

扩散模型每一步去噪都需要让网络**知道当前噪声水平 σ**。但 σ 是一个标量，怎么把它喂给一个吃 (B, N, c) 张量的 Transformer? 标准做法是 sinusoidal /random Fourier embedding，把 σ 映成一个 c 维向量。

### 数学背景: random feature map

Random Fourier Features ([Rahimi & Recht 2007](https://people.eecs.berkeley.edu/~brecht/papers/07.rah.rec.nips.pdf))证明了一个关键事实: **平移不变核** $k(t, t') = k(t - t')$ 可以通过随机三角函数特征近似:

$$k(t, t') \approx \phi(t)^\top \phi(t'), \quad \phi(t) = \sqrt{2/c} \,\big[\cos(w_1 t + b_1), \dots, \cos(w_c t + b_c)\big]$$

其中 $w_k \sim p(w)$ (Fourier 谱)、$b_k \sim U(0, 2\pi)$。**c 越大近似越准确**，且 $\phi(t)$ 之间的内积就是核函数。

AF3 借用这个想法 (不再要求核近似严格)，把噪声水平 $\tau = \log(\sigma / \sigma_\text{data})$(注意是 log) 映成 $c_\text{noise}$ 维稠密向量:

$$\mathrm{FourierEmbed}(\tau)_k = \cos\big(2 \pi \,(\tau \cdot w_k + b_k)\big), \quad k = 1 \ldots c$$

$w_k, b_k$ 是构造时一次性抽样的高斯/均匀随机数，作为**不可训练的 nn.Parameter** 存进 state_dict。

### 为什么用 log(σ/σ_data) 而不是 σ 本身

扩散里 σ 跨越多个数量级 (典型 $\sigma_\text{min} \approx 0.002, \sigma_\text{max} \approx 80$，AF3 范围更广)。如果直接用 σ:

- σ ≈ 0 时几乎所有 $w_k \cdot \sigma + b_k \approx b_k$，cos 值全相同 —— 不同 σ 撞码
- σ 很大时 cos 振荡极快、相邻 σ 的 embedding 完全无关 —— 学不到平滑结构

**取 log**: σ 跨越 ~4 个数量级时 log 只跨 9 倍，cos 在合理频段振荡 ——
embedding 既能区分远端 σ、又对相邻 σ 平滑。

### 为什么是 cos 而不是 sin + cos / position encoding?

AF3 直接用 cos —— 因为 $\cos(\phi - \pi/2) = \sin(\phi)$，加上随机相位 $b_k$已经隐含覆盖了 sin/cos 两个相位。c 维 "随机方向" 的 cos 值合起来就是一个稠密可分辨的 fingerprint，让网络容易区分不同 σ。

另一个选择是 Transformer 经典的 sinusoidal positional encoding (对数频率序列)。AF3 用 random feature 而非固定频率 —— 让模型不依赖人工设计的频率层次，靠数据决定哪个 $w_k$ 重要。

### 它在哪里被用到

唯一调用方: `DiffusionConditioning` (第 4 章) 里把 `t_hat / sigma_data` 取对数再过 FourierEmbedding，得到一个 c 维向量、LN + Linear 后加到单序列条件 `single_s` 上。这是把噪声水平注入 AdaLN-Zero 的路径。

**任务**: 在同一个文件里填 `FourierEmbedding.__init__` 和 `.forward`。注意 `__init__` 用一个 manual_seed 的 generator 才能让权重每次构造一致 ——测试 harness 会再把它们覆盖成 linspace。

In [ ]:
from feature_embedding.relative_position_encoding import FourierEmbedding
from feature_embedding.control_values.feature_embedding_checks import c_noise

fe = FourierEmbedding(c=c_noise)
test_module_shape(fe, 'fourier_embedding', control_folder)
test_module_forward(
    fe, 'fourier_embedding',
    inputs=(test_inputs['noise_level'],),
    output_names='out',
    control_folder=control_folder,
)
print('FourierEmbedding ✓')

## 章节小结

本章你交付了两个小而关键的零件:

1. **`RelativePositionEncoding`** —— 算法 3。给 Pairformer 和 DiffusionConditioning   提供"两个 token 在结构空间的相对偏移"先验。
2. **`FourierEmbedding`** —— 算法 22。给 DiffusionConditioning 提供"当前噪声水平的   稠密表示"，是 AdaLN-Zero 让 transformer 感知 σ 的唯一通道。

本章对其它 embedder (InputFeatureEmbedder / AtomAttentionEncoder /ConstraintEmbedder) 留了 TODO 但没单测 —— 它们依赖整个 `input_feature_dict`，在端到端 `overview.ipynb` 里一并验证。

**下一站**: 第 4 章 Diffusion 会大量用到本章的 FourierEmbedding (在DiffusionConditioning 里) 和 RelativePositionEncoding (作为 pair 条件起点)。